# L7: 无监督学习算法


# L7: 无监督学习算法

**课时**: 2 课时（90 分钟/课时）

## 1. 学习目标

完成本课程后，学员能够：
1. 解释K-Means聚类的目标函数与EM算法思想
2. 理解PCA的降维原理与主成分选择标准
3. 掌握异常检测的统计方法与Isolation Forest
4. 了解推荐系统的协同过滤与矩阵分解方法
5. 使用sklearn实现无监督学习的完整流程

## 2. 核心知识点

### 2.1 K-Means聚类

**目标**: 最小化簇内平方和 (WCSS)

**算法流程**:
1. 随机初始化K个簇中心
2. E步：将每个点分配给最近的簇中心
3. M步：更新簇中心为该簇所有点的均值
4. 重复2-3直至收敛


In [ ]:
from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt

X, y_true = make_blobs(n_samples=500, centers=4, cluster_std=1.0, random_state=42)

# K-Means++初始化
kmeans = KMeans(n_clusters=4, init='k-means++', n_init=10, random_state=42)
labels = kmeans.fit_predict(X)
print(f"轮廓系数: {silhouette_score(X, labels):.3f}")

# 确定最优K：肘部法则
inertias, silhouette_scores = [], []
K_range = range(2, 10)
for k in K_range:
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    km.fit(X)
    inertias.append(km.inertia_)
    silhouette_scores.append(silhouette_score(X, km.labels_))


### 2.2 PCA（主成分分析）

**目标**: 找到数据方差最大的正交方向，将高维数据投影到低维空间

**方差解释率**: 前k个主成分解释的总方差比例为 ∑ᵢ₌₁ᵏλᵢ / ∑ⱼλⱼ


In [ ]:
from sklearn.decomposition import PCA
from sklearn.datasets import load_iris

iris = load_iris()
X = iris.data

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)

print(f"各主成分解释方差比例: {pca.explained_variance_ratio_}")
print(f"累计解释方差: {sum(pca.explained_variance_ratio_):.3f}")

# 累积解释方差
pca_full = PCA().fit(X)
cumsum = np.cumsum(pca_full.explained_variance_ratio_)
plt.plot(range(1, 5), cumsum, 'bo-')
plt.axhline(y=0.95, color='r', linestyle='--', label='95%阈值')
plt.xlabel("主成分数量")
plt.ylabel("累计解释方差")
plt.savefig("l7_pca.png", dpi=150)
plt.show()


### 2.3 异常检测

**方法分类**:
1. **统计方法**: 假设数据服从正态分布，检测偏离分布的点
2. **Isolation Forest**: 随机切分数据，异常点更容易被隔离
3. **One-Class SVM**: 学习正常数据的边界


In [ ]:
from sklearn.ensemble import IsolationForest

# 生成带异常点的数据
outliers_fraction = 0.05
# ... 数据生成 ...

iforest = IsolationForest(contamination=outliers_fraction, n_estimators=100, random_state=42)
y_pred = iforest.fit_predict(X)
anomaly_detected = (y_pred == -1).sum()
print(f"检测到 {anomaly_detected} 个异常（占比 {anomaly_detected/len(X):.1%}）")


### 2.4 推荐系统基础

**矩阵分解 (SVD/MF)**:
- 将用户-物品评分矩阵 R (m×n) 分解为用户矩阵 U (m×k) 和物品矩阵 V (n×k)ᵀ
- R ≈ U·Vᵀ


In [ ]:
import numpy as np

n_users, n_items = 100, 50
R = np.random.randint(0, 6, (n_users, n_items))

from numpy.linalg import svd
U, sigma, Vt = svd(R, full_matrices=False)

k = 10
U_k = U[:, :k]
sigma_k = np.diag(sigma[:k])
V_k = Vt[:k, :]
R_pred = np.dot(np.dot(U_k, sigma_k), V_k)
print(f"预测评分矩阵形状: {R_pred.shape}")


## 3. 代码示例


In [ ]:
"""
L7-unsupervised-learning.py
综合演示：K-Means、PCA、异常检测的联合应用
"""

import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.metrics import silhouette_score

# 场景：电商用户行为聚类 + 异常检测
print("电商用户行为分析：聚类 + 异常检测")

np.random.seed(42)
n_normal, n_anomaly = 450, 50

normal_data = np.random.normal(loc=[50, 10, 20, 5], scale=[15, 5, 8, 2], size=(n_normal, 4))
anomaly_data = np.random.uniform(low=[0, 0, 0, 0], high=[200, 50, 100, 30], size=(n_anomaly, 4))

X = np.vstack([normal_data, anomaly_data])
y_true = np.concatenate([np.zeros(n_normal), np.ones(n_anomaly)])

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 确定最优聚类数
inertias, silhouettes = [], []
K_range = range(2, 8)
for k in K_range:
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    km.fit(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, km.labels_))

optimal_k = K_range[np.argmax(silhouettes)]
print(f"最优K值（轮廓系数）: {optimal_k}")

km_final = KMeans(n_clusters=optimal_k, n_init=10, random_state=42)
labels = km_final.fit_predict(X_scaled)

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

iforest = IsolationForest(contamination=0.1, n_estimators=100, random_state=42)
anomaly_labels = iforest.fit_predict(X_scaled)

# 可视化
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

axes[0, 0].plot(K_range, inertias, 'bo-')
axes[0, 0].axvline(x=optimal_k, color='r', linestyle='--')
axes[0, 0].set_title('肘部法则')

scatter = axes[0, 1].scatter(X_pca[:, 0], X_pca[:, 1], c=y_true, cmap='RdYlGn', alpha=0.6)
axes[0, 1].set_title('PCA（真实标签）')

scatter2 = axes[1, 0].scatter(X_pca[:, 0], X_pca[:, 1], c=labels, cmap='tab10', alpha=0.6)
axes[1, 0].set_title(f'PCA（K-Means K={optimal_k}）')

colors = ['green' if a == 1 else 'red' for a in anomaly_labels]
axes[1, 1].scatter(X_pca[:, 0], X_pca[:, 1], c=colors, alpha=0.5)
axes[1, 1].set_title('Isolation Forest异常检测（绿=正常，红=异常）')

plt.suptitle("L7 综合实战：电商用户行为分析", fontsize=14)
plt.tight_layout()
plt.savefig("l7_ecommerce_analysis.png", dpi=150, bbox_inches='tight')
plt.show()


## 4. 练习题

1. **K-Means实现**: 不使用sklearn，自己实现K-Means算法（含K-Means++初始化）。
2. **PCA推导**: 手动推导PCA数学原理，解释为什么选择最大特征值对应的特征向量。
3. **异常检测实验**: 比较EllipticEnvelope、IsolationForest、One-ClassSVM的检测效果。
4. **聚类评估**: 使用make_moons数据比较K-Means、层次聚类、DBSCAN的效果。
5. **综合应用**: 对葡萄酒数据集进行PCA降维+聚类分析，解释聚类结果的含义。

## 5. 延伸阅读

- **书籍**: Charu Aggarwal, *Outlier Analysis* — 异常检测领域权威教材
- **论文**: MacQueen, 1967, "Some Methods for Classification and Analysis of Multivariate Observations" — K-Means原始论文
- **工具**: PyOD库 — 专门用于异常检测的Python库 https://pyod.readthedocs.io/

---
